# Ego Gate offline benchmark expansion
Private GPU execution. Synthetic/unit tests are separate from benchmark evidence.
Fixed protocol; preserve all outcomes.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib
import torch
base=Path("/kaggle/working/benchmark")
base.mkdir(exist_ok=True)
assert torch.cuda.is_available(), "GPU unavailable"
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],flush=True)
os.environ["OMP_NUM_THREADS"]="2"
os.environ["MKL_NUM_THREADS"]="2"
os.environ["OPENBLAS_NUM_THREADS"]="2"


In [ ]:
(base/"offline_benchmark.py").write_text('"""Fixed-protocol, GPU offline selection benchmark; see BENCHMARK_PROTOCOL.md."""\nfrom __future__ import annotations\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport os\nimport platform\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.func import functional_call, grad, vmap\nfrom sklearn.datasets import load_digits\nfrom sklearn.model_selection import train_test_split\n\nMETHODS = (\'random\', \'stratified_random\', \'doubt\', \'curiosity\', \'egogate\',\n           \'embedding_kcenter\', \'egogate_diverse\', \'loss\', \'herding\')\n\n\nclass MLP(nn.Module):\n    def __init__(self, dim, hidden):\n        super().__init__()\n        self.fc1, self.fc2 = nn.Linear(dim, hidden), nn.Linear(hidden, 10)\n\n    def forward(self, x, features=False):\n        h = F.relu(self.fc1(x.flatten(1)))\n        return (self.fc2(h), h) if features else self.fc2(h)\n\n\nclass CNN(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.conv = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),\n                                  nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),\n                                  nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))\n        self.fc1, self.fc2 = nn.Linear(128 * 4 * 4, 128), nn.Linear(128, 10)\n\n    def forward(self, x, features=False):\n        h = F.relu(self.fc1(self.conv(x).flatten(1)))\n        return (self.fc2(h), h) if features else self.fc2(h)\n\n\ndef settings(dataset):\n    if dataset == \'digits\':\n        return dict(lr=.5, momentum=0., epochs_a=60, epochs_b=40, batch=32,\n                    replay=2, budgets=[20, 40, 100])\n    return dict(lr=.05 if dataset == \'cifar10\' else .1,\n                momentum=.9 if dataset == \'cifar10\' else 0.,\n                epochs_a=20 if dataset == \'cifar10\' else 10,\n                epochs_b=20 if dataset == \'cifar10\' else 10,\n                batch=128, replay=8, budgets=[50, 200, 500])\n\n\ndef load_data(name, root, seed):\n    if name == \'digits\':\n        d = load_digits()\n        ix, it = train_test_split(np.arange(len(d.target)), test_size=.2,\n                                 stratify=d.target, random_state=seed)\n        return (torch.tensor(d.data[ix] / 16, dtype=torch.float32), torch.tensor(d.target[ix]),\n                torch.tensor(d.data[it] / 16, dtype=torch.float32), torch.tensor(d.target[it]))\n    from torchvision.datasets import FashionMNIST, CIFAR10\n    cls = CIFAR10 if name == \'cifar10\' else FashionMNIST\n    download = True\n    if name == \'cifar10\' and Path(\'/kaggle/input\').exists():\n        mirrors = sorted(Path(\'/kaggle/input\').glob(\'**/cifar-10-batches-py\'))\n        if mirrors:\n            # torchvision validates the official per-batch MD5 checksums.\n            root, download = mirrors[0].parent, False\n    train, test = cls(root, train=True, download=download), cls(root, train=False, download=download)\n    def convert(d):\n        x = torch.as_tensor(np.asarray(d.data).copy(), dtype=torch.float32) / 255\n        if name == \'cifar10\':\n            x = x.permute(0, 3, 1, 2).contiguous()\n            x = (x - torch.tensor([.4914, .4822, .4465])[None, :, None, None]) / torch.tensor([.247, .243, .261])[None, :, None, None]\n        return x, torch.as_tensor(d.targets, dtype=torch.long)\n    return (*convert(train), *convert(test))\n\n\ndef sync(device):\n    if str(device).startswith(\'cuda\'):\n        torch.cuda.synchronize(device)\n\n\ndef train(model, x, y, cfg, seed, epochs, rx=None, ry=None):\n    model.train()\n    opt = torch.optim.SGD(model.parameters(), lr=cfg[\'lr\'], momentum=cfg[\'momentum\'])\n    # Separate generators prevent replay policy from altering new-example order.\n    order_rng = torch.Generator().manual_seed(seed + 20000)\n    replay_rng = torch.Generator().manual_seed(seed + 30000)\n    updates = exposures = replay_exposures = 0\n    for epoch in range(epochs):\n        order = torch.randperm(len(y), generator=order_rng)\n        for start in range(0, len(y), cfg[\'batch\']):\n            ix = order[start:start + cfg[\'batch\']].to(y.device)\n            bx, by = x[ix], y[ix]\n            if rx is not None and len(rx):\n                ri = torch.randint(len(rx), (cfg[\'replay\'],), generator=replay_rng).to(y.device)\n                bx, by = torch.cat((bx, rx[ri])), torch.cat((by, ry[ri]))\n                replay_exposures += len(ri)\n            opt.zero_grad(set_to_none=True)\n            F.cross_entropy(model(bx), by).backward()\n            opt.step()\n            updates += 1\n            exposures += len(by)\n    return dict(updates=updates, exposures=exposures, replay_exposures=replay_exposures)\n\n\n@torch.no_grad()\ndef accuracy(model, x, y):\n    model.eval()\n    correct = torch.zeros((), device=y.device)\n    for start in range(0, len(y), 512):\n        correct += (model(x[start:start+512]).argmax(1) == y[start:start+512]).sum()\n    return float(correct.item() / len(y))\n\n\ndef score_pool(model, x, y, mode):\n    """Only compute signals needed by the selector, charging their actual cost."""\n    model.eval()\n    want_grad = mode in (\'curiosity\', \'egogate\', \'egogate_diverse\')\n    want_embed = mode in (\'embedding_kcenter\', \'egogate_diverse\', \'herding\')\n    parts = {k: [] for k in (\'doubt\', \'curiosity\', \'embedding\', \'loss\')}\n    params = {k: v.detach() for k, v in model.named_parameters()}\n    def single_loss(p, sample, label):\n        return F.cross_entropy(functional_call(model, p, (sample.unsqueeze(0),)), label.unsqueeze(0))\n    per_grad = vmap(grad(single_loss), in_dims=(None, 0, 0))\n    batch = 16 if isinstance(model, CNN) and want_grad else 512\n    for start in range(0, len(y), batch):\n        bx, by = x[start:start+batch], y[start:start+batch]\n        with torch.no_grad():\n            logits, h = model(bx, features=True)\n            p = logits.softmax(1)\n            parts[\'doubt\'].append((-(p * p.clamp_min(1e-12).log()).sum(1)).cpu())\n            parts[\'loss\'].append(F.cross_entropy(logits, by, reduction=\'none\').cpu())\n            if want_embed:\n                parts[\'embedding\'].append(h.cpu())\n            if want_grad and isinstance(model, MLP):\n                err = p.clone()\n                err[torch.arange(len(by), device=by.device), by] -= 1\n                back = (err @ model.fc2.weight) * (h > 0)\n                g2 = err.square().sum(1) * (h.square().sum(1) + 1)\n                g2 += back.square().sum(1) * (bx.flatten(1).square().sum(1) + 1)\n                parts[\'curiosity\'].append(g2.cpu())\n        if want_grad and isinstance(model, CNN):\n            gradients = per_grad(params, bx, by)\n            norm = sum(v.flatten(1).square().sum(1) for v in gradients.values())\n            parts[\'curiosity\'].append(norm.detach().cpu())\n            del gradients\n    out = {k: torch.cat(v).numpy() for k, v in parts.items() if v}\n    if want_grad:\n        def z(v):\n            return (v - v.mean()) / max(float(v.std()), 1e-12)\n        out[\'egogate\'] = z(out[\'doubt\']) + z(out[\'curiosity\'])\n    return out\n\n\ndef kcenter(features, k, candidates=None):\n    pool = np.arange(len(features)) if candidates is None else candidates\n    z = features[pool].astype(np.float64)\n    z = (z - z.mean(0)) / np.maximum(z.std(0), 1e-8)\n    chosen = [int((z*z).sum(1).argmax())]\n    distance = ((z - z[chosen[0]])**2).sum(1)\n    distance[chosen] = -1\n    for _ in range(1, k):\n        idx = int(distance.argmax())\n        chosen.append(idx)\n        distance = np.minimum(distance, ((z-z[idx])**2).sum(1))\n        distance[chosen] = -1\n    return pool[np.asarray(chosen)]\n\n\ndef herding(features, labels, k):\n    """Balanced herding adaptation: L2-normalized features, no duplicate exemplars.\n\n    Greedily minimize distance of the selected arithmetic mean to the class mean.\n    This is a component baseline, not full iCaRL or its nearest-mean classifier.\n    """\n    result = []\n    classes = np.unique(labels)\n    for pos, label in enumerate(classes):\n        pool = np.flatnonzero(labels == label)\n        z = features[pool].astype(np.float64)\n        z /= np.maximum(np.linalg.norm(z, axis=1, keepdims=True), 1e-12)\n        target, running = z.mean(0), np.zeros(z.shape[1])\n        used = np.zeros(len(pool), dtype=bool)\n        count = k // len(classes) + int(pos < k % len(classes))\n        for i in range(count):\n            distances = ((target - (z + running)/(i+1))**2).sum(1)\n            distances[used] = np.inf\n            nxt = int(distances.argmin())\n            used[nxt] = True\n            running += z[nxt]\n            result.append(pool[nxt])\n    return np.asarray(result, dtype=np.int64)\n\n\ndef select(model, x, y, method, k, seed):\n    rng = np.random.default_rng(seed + 40000)\n    labels = y.cpu().numpy()\n    if method == \'none\':\n        return np.empty(0, dtype=np.int64)\n    if method == \'full\':\n        return np.arange(len(y))\n    if method == \'random\':\n        return rng.choice(len(y), k, replace=False)\n    if method == \'stratified_random\':\n        classes = np.unique(labels)\n        return np.concatenate([rng.choice(np.flatnonzero(labels == c),\n                               k//len(classes) + int(i < k % len(classes)), replace=False)\n                               for i, c in enumerate(classes)])\n    scores = score_pool(model, x, y, method)\n    if method == \'herding\':\n        return herding(scores[\'embedding\'], labels, k)\n    if method == \'embedding_kcenter\':\n        return kcenter(scores[\'embedding\'], k)\n    if method == \'egogate_diverse\':\n        candidates = np.argsort(scores[\'egogate\'], kind=\'stable\')[-min(len(y), 5*k):]\n        return kcenter(scores[\'embedding\'], k, candidates)\n    return np.argsort(scores[method], kind=\'stable\')[-k:][::-1].copy()\n\n\ndef run_seed(dataset, seed, output, root, device, smoke=False):\n    cfg = settings(dataset)\n    if smoke:\n        cfg.update(epochs_a=1, epochs_b=1, budgets=[10])\n    # Never silently skip a partial seed on resume.\n    path = output / f\'{dataset}_{seed}.json\'\n    if path.exists():\n        raise FileExistsError(f\'Refusing to overwrite existing evidence: {path}\')\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    x, y, tx, ty = load_data(dataset, root, seed)\n    classes = np.random.default_rng(seed + 50000).permutation(10)\n    a = torch.isin(y, torch.tensor(classes[:5]))\n    ta = torch.isin(ty, torch.tensor(classes[:5]))\n    arrays = [x[a], y[a], x[~a], y[~a], tx[ta], ty[ta], tx[~ta], ty[~ta]]\n    if smoke:\n        arrays = [v[:256] for v in arrays]\n    xa, ya, xb, yb, xta, yta, xtb, ytb = [v.to(device) for v in arrays]\n    del x, y, tx, ty, arrays\n    model = (CNN() if dataset == \'cifar10\' else MLP(xa[0].numel(), 64 if dataset == \'digits\' else 128)).to(device)\n    sync(device)\n    start = time.perf_counter()\n    train(model, xa, ya, cfg, seed, cfg[\'epochs_a\'])\n    sync(device)\n    base_seconds = time.perf_counter() - start\n    before = accuracy(model, xta, yta)\n    rows = []\n    jobs = [(budget, method) for budget in cfg[\'budgets\'] for method in METHODS]\n    jobs += [(0, \'none\'), (len(ya), \'full\')]\n    # Randomized condition execution order reduces systematic timing-order bias.\n    np.random.default_rng(seed + 60000).shuffle(jobs)\n    for budget, method in jobs:\n        candidate = copy.deepcopy(model)\n        sync(device)\n        if str(device).startswith(\'cuda\'):\n            torch.cuda.reset_peak_memory_stats(device)\n        start = time.perf_counter()\n        idx = select(model, xa, ya, method, budget, seed)\n        sync(device)\n        selection_seconds = time.perf_counter() - start\n        assert len(idx) == budget and len(np.unique(idx)) == budget\n        selected = torch.as_tensor(idx, device=device)\n        rx, ry = xa[selected], ya[selected]\n        sync(device)\n        start = time.perf_counter()\n        counts = train(candidate, xb, yb, cfg, seed, cfg[\'epochs_b\'], rx, ry)\n        sync(device)\n        training_seconds = time.perf_counter() - start\n        after, new = accuracy(candidate, xta, yta), accuracy(candidate, xtb, ytb)\n        row = dict(dataset=dataset, seed=seed, budget=budget, method=method,\n                   train_a_count=len(ya), train_b_count=len(yb),\n                   test_a_count=len(yta), test_b_count=len(ytb),\n                   parameter_count=sum(p.numel() for p in model.parameters()),\n                   task_a_pre=before, task_a_post=after, task_b_post=new,\n                   final_accuracy=(after+new)/2, forgetting=before-after,\n                   selection_seconds=selection_seconds, training_seconds=training_seconds,\n                   base_seconds=base_seconds, memory_bytes=rx.numel()*rx.element_size()+ry.numel()*ry.element_size(),\n                   cuda_peak_bytes=torch.cuda.max_memory_allocated(device) if str(device).startswith(\'cuda\') else None,\n                   coverage=int(len(torch.unique(ry))), class_counts=torch.bincount(ry, minlength=10).cpu().tolist(),\n                   selection_sha256=hashlib.sha256(np.asarray(idx, dtype=\'<i8\').tobytes()).hexdigest(),\n                   selected_indices=idx.tolist(), **counts)\n        rows.append(row)\n        (output / f\'{dataset}_{seed}.partial.json\').write_text(json.dumps(rows, indent=2))\n        print(json.dumps({k: row[k] for k in (\'dataset\',\'seed\',\'budget\',\'method\',\'final_accuracy\',\'selection_seconds\',\'training_seconds\')}), flush=True)\n        del candidate, rx, ry\n    path.write_text(json.dumps(dict(config=cfg, classes=classes.tolist(), seed=seed,\n                                    dataset=dataset, rows=rows, complete=True), indent=2))\n    (output / f\'{dataset}_{seed}.partial.json\').unlink()\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'--dataset\', choices=[\'digits\', \'fashion\', \'cifar10\'], required=True)\n    p.add_argument(\'--seeds\', nargs=\'+\', type=int, default=list(range(100,110)))\n    p.add_argument(\'--output\', type=Path, required=True)\n    p.add_argument(\'--data\', type=Path, default=Path(\'/kaggle/working/data\'))\n    p.add_argument(\'--device\', default=\'cuda:0\')\n    p.add_argument(\'--smoke\', action=\'store_true\')\n    args = p.parse_args()\n    if args.device.startswith(\'cuda\') and not torch.cuda.is_available():\n        raise RuntimeError(\'Requested Kaggle GPU is unavailable; no silent CPU fallback\')\n    torch.set_num_threads(2)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    args.output.mkdir(parents=True, exist_ok=True)\n    import torchvision\n    import scipy\n    import sklearn\n    manifest = dict(python=platform.python_version(), torch=torch.__version__,\n                    torchvision=torchvision.__version__, numpy=np.__version__,\n                    scipy=scipy.__version__, sklearn=sklearn.__version__,\n                    cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),\n                    device=args.device, gpu=torch.cuda.get_device_name(args.device) if torch.cuda.is_available() else None,\n                    platform=platform.platform(), cpu=platform.processor(), threads=torch.get_num_threads(),\n                    gpu_memory_bytes=torch.cuda.get_device_properties(args.device).total_memory if torch.cuda.is_available() else None,\n                    source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n                    cifar_mirror=\'pankrzysiu/cifar10-python; torchvision official MD5 validation\',\n                    seeds=args.seeds, dataset=args.dataset, smoke=args.smoke,\n                    kaggle=bool(os.environ.get(\'KAGGLE_KERNEL_RUN_TYPE\')))\n    (args.output/\'manifest.json\').write_text(json.dumps(manifest, indent=2))\n    for seed in args.seeds:\n        run_seed(args.dataset, seed, args.output, args.data, args.device, args.smoke)\n    (args.output/\'COMPLETE.json\').write_text(json.dumps(manifest, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n',encoding="utf-8")
(base/"BENCHMARK_PROTOCOL.md").write_text("# Additional benchmark protocol (fixed before new outcomes)\n\nThis study strengthens the current paper's **offline storage selection** evidence.\nIt does not redesign the online controller or replace earlier negative results.\nIt is a dated local protocol, not an externally registered preregistration.\n\n## Questions and design\n\nDoes the original sum of standardized entropy and squared gradient norm improve\nretention and final accuracy beyond random, entropy alone, and representative\nselection? How does the answer depend on dataset and memory budget?\n\n- Datasets: Digits (stratified 80/20 split per seed), full official Fashion-MNIST\n  and CIFAR-10 train/test splits. No leaderboard scores imported as comparisons.\n  CIFAR is attached from Kaggle's `pankrzysiu/cifar10-python` mirror; torchvision\n  checks the official batch-file MD5 hashes. The public source is a data mirror,\n  not a source of baseline performance numbers.\n- Ten paired seeds 100--109. Each seed randomizes the five-class/five-class\n  partition, initialization, training order, and random selection. No task label\n  or class mask is supplied at inference; predictions always use ten outputs.\n- Digits retains the original 64-hidden-unit MLP, SGD 0.5, 60/40 epochs,\n  batches of 32 new and 2 replay examples. Memory sizes 20, 40, 100.\n- Fashion uses a 128-hidden-unit MLP, SGD 0.1, 10/10 epochs, batches of 128\n  new and 8 replay examples. Memory sizes 50, 200, 500.\n- CIFAR uses a three-block convolutional network without batch normalization,\n  SGD 0.05 with momentum 0.9, 20/20 epochs, batches of 128 new and 8 replay\n  examples, normalized pixels, no augmentation. Memory sizes 50, 200, 500.\n  This is a transparent controlled CNN experiment, not an optimized CIFAR entry.\n- Fixed configurations, no test-driven tuning or winner-only reporting. Any\n  amendment must be recorded explicitly before additional runs.\n- Nine bounded selectors: random, class-balanced random, entropy, gradient\n  norm, original EgoGate, embedding k-center, EgoGate-diverse (existing rule),\n  high training loss, and class-balanced herding. Herding is an adaptation of\n  the exemplar component of iCaRL, not the complete iCaRL learner. Full-pool\n  and no-replay anchors are run once per seed, separately from bounded methods.\n- Base Task-A checkpoints are shared within a seed. Task-B training order and\n  replay-position random streams are independent and shared across selectors.\n  This removes the archived code's coupling between shuffling and replay RNG.\n- Score normalization is global over the Task-A training pool, as in the\n  original offline rule. CNN gradient norms use exact per-example autograd.\n- Equal update and replay exposure budgets, plus actual selection and training\n  wall times, persistent exemplar bytes, CUDA peak allocations, selected-class\n  counts, and hashes of selected indices. Timing includes synchronization.\n  Selection may access the entire training pool: this is not bounded online RAM.\n\n## Inference\n\nPrimary endpoint: final equally weighted mean of Task-A and Task-B accuracy.\nPrimary contrasts: EgoGate minus random, entropy, and herding in each of the\nnine dataset/budget cells (27 tests, a single Holm family). Report seed-paired\n95% t intervals, all raw values, and sample standard deviations. Intervals are\npointwise, not multiplicity-adjusted. Seeds vary several nuisance factors and\nare not independent datasets. Forgetting, new-task accuracy, other selectors,\ncost, and coverage are secondary/descriptive. No significance-based stopping.\n\nExpected bounded rows: 3 datasets x 3 budgets x 10 seeds x 9 selectors = 810.\nExpected anchor rows: 3 datasets x 10 seeds x 2 = 60. Total: 870.\n\nGPU execution must be verified in the environment manifest. Tests and a\nnon-inferential execution check can use synthetic inputs locally. Genuine\nbenchmark evidence must come from Kaggle. Preserve source hashes and raw rows;\nnever label a partial run complete.\n\n## Limits even after completion\n\nThis does not establish state of the art, general AGI progress, efficacy in\nagent memory, or superiority to complete GSS/GCR/MIR/GRASP systems. The added\npublished-component baseline is herding. Multi-phase state-of-the-art systems,\narchitecture sweeps, and an independently reviewed release remain further work.\n",encoding="utf-8")


In [ ]:
(base/"analyze_offline_benchmark.py").write_text('"""Validate the entire fixed benchmark grid before producing inferential output."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom scipy import stats\n\nMETHODS=(\'random\',\'stratified_random\',\'doubt\',\'curiosity\',\'egogate\',\'embedding_kcenter\',\'egogate_diverse\',\'loss\',\'herding\')\nBUDGETS={\'digits\':[20,40,100],\'fashion\':[50,200,500],\'cifar10\':[50,200,500]}\n\n\ndef write_csv(path, rows):\n    with path.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n        writer=csv.DictWriter(f,fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef analyze(root, output):\n    rows=[]\n    manifests=list(root.glob(\'**/manifest.json\'))\n    if not manifests:\n        raise ValueError(\'No execution manifests\')\n    hashes=set()\n    for file in manifests:\n        m=json.loads(file.read_text())\n        assert m[\'gpu\'] and m[\'device\'].startswith(\'cuda\') and not m[\'smoke\']\n        assert (file.parent/\'COMPLETE.json\').exists(), f\'Incomplete shard: {file}\'\n        hashes.add(m[\'source_sha256\'])\n    assert len(hashes)==1, \'Mixed source versions\'\n    for dataset in BUDGETS:\n        for file in root.glob(f\'**/{dataset}_*.json\'):\n            if \'.partial.\' in file.name:\n                raise ValueError(f\'Incomplete evidence: {file}\')\n            doc=json.loads(file.read_text())\n            assert doc[\'complete\'] and doc[\'dataset\']==dataset\n            rows.extend(doc[\'rows\'])\n    keys=[(r[\'dataset\'],r[\'seed\'],r[\'budget\'],r[\'method\']) for r in rows]\n    assert len(keys)==len(set(keys)), \'Duplicate evidence\'\n    expected={(d,s,b,m) for d,bs in BUDGETS.items() for s in range(100,110) for b in bs for m in METHODS}\n    bounded=[r for r in rows if r[\'method\'] in METHODS]\n    assert {k for k in keys if k[3] in METHODS}==expected, \'Incomplete bounded grid\'\n    anchors=[r for r in rows if r[\'method\'] not in METHODS]\n    assert len(anchors)==60 and {(r[\'dataset\'],r[\'seed\'],r[\'method\']) for r in anchors}=={\n        (d,s,m) for d in BUDGETS for s in range(100,110) for m in (\'none\',\'full\')}\n    for d in BUDGETS:\n        for s in range(100,110):\n            group=[r for r in bounded if r[\'dataset\']==d and r[\'seed\']==s]\n            assert len({(r[\'updates\'],r[\'exposures\'],r[\'replay_exposures\']) for r in group})==1\n            assert len({r[\'task_a_pre\'] for r in group})==1\n    for r in rows:\n        ix=r[\'selected_indices\']\n        assert len(ix)==r[\'budget\']==len(set(ix))\n        assert sum(r[\'class_counts\'])==r[\'budget\']\n        assert hashlib.sha256(np.asarray(ix,dtype=\'<i8\').tobytes()).hexdigest()==r[\'selection_sha256\']\n        assert all(np.isfinite(r[k]) for k in (\'final_accuracy\',\'forgetting\',\'selection_seconds\',\'training_seconds\'))\n        assert abs(r[\'final_accuracy\']-(r[\'task_a_post\']+r[\'task_b_post\'])/2)<1e-10\n    output.mkdir(parents=True,exist_ok=True)\n    flat=[{k:v for k,v in r.items() if k not in (\'selected_indices\',\'class_counts\')} for r in rows]\n    write_csv(output/\'per_run.csv\',flat)\n    summary=[]\n    for d,bs in BUDGETS.items():\n        for b in bs:\n            for m in METHODS:\n                group=[r for r in bounded if (r[\'dataset\'],r[\'budget\'],r[\'method\'])==(d,b,m)]\n                row=dict(dataset=d,budget=b,method=m,n=len(group))\n                for metric in (\'final_accuracy\',\'forgetting\',\'task_a_pre\',\'task_a_post\',\'task_b_post\',\'selection_seconds\',\'training_seconds\',\'coverage\',\'memory_bytes\',\'cuda_peak_bytes\'):\n                    values=np.asarray([r[metric] for r in group])\n                    row[metric+\'_mean\']=float(values.mean())\n                    row[metric+\'_std\']=float(values.std(ddof=1))\n                summary.append(row)\n    write_csv(output/\'summary.csv\',summary)\n    contrasts=[]\n    for d,bs in BUDGETS.items():\n        for b in bs:\n            for comparator in (\'random\',\'doubt\',\'herding\'):\n                def values(m):\n                    return np.asarray([next(r[\'final_accuracy\'] for r in bounded if (r[\'dataset\'],r[\'budget\'],r[\'method\'],r[\'seed\'])==(d,b,m,s)) for s in range(100,110)])\n                diff=100*(values(\'egogate\')-values(comparator))\n                se=stats.sem(diff)\n                ci=stats.t.ppf(.975,9)*se\n                p=float(stats.ttest_1samp(diff,0).pvalue) if se else (1. if diff.mean()==0 else 0.)\n                contrasts.append(dict(dataset=d,budget=b,comparator=comparator,n=10,\n                                      accuracy_delta_pp=float(diff.mean()),ci_low=float(diff.mean()-ci),ci_high=float(diff.mean()+ci),p_value=p))\n    running=0.\n    for rank,i in enumerate(sorted(range(len(contrasts)),key=lambda i:contrasts[i][\'p_value\'])):\n        running=max(running,(len(contrasts)-rank)*contrasts[i][\'p_value\'])\n        contrasts[i][\'holm_p_value\']=min(running,1.)\n    write_csv(output/\'primary_comparisons.csv\',contrasts)\n    audit=dict(complete=True,rows=len(rows),bounded_rows=len(bounded),anchor_rows=len(anchors),\n               primary_tests=len(contrasts),source_sha256=list(hashes)[0],gpu_manifests=len(manifests))\n    (output/\'audit.json\').write_text(json.dumps(audit,indent=2))\n    print(json.dumps(audit,indent=2))\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser()\n    p.add_argument(\'root\',type=Path)\n    p.add_argument(\'--output\',type=Path,required=True)\n    a=p.parse_args()\n    analyze(a.root,a.output)\n',encoding="utf-8")


In [ ]:
manifest={'source_sha256': '13e9b77ebcfe631cdc161ad48617a958859faf199ef99d0fbcd6ac14ae25b0c7', 'protocol_sha256': '62e01882fd0e80de76159c63e7a86171b9b519c3ee8f1adb8196d62c962b3e64', 'analysis_sha256': 'f53d7256f3d16393161851bb961e13e4413a3a43023b261ac648727160c5609a', 'smoke': False}
(base/"source_manifest.json").write_text(json.dumps(manifest,indent=2))


In [ ]:
sys.path.insert(0,str(base))
from offline_benchmark import load_data
# Download/extract once, before workers, to avoid racing over shared archives.
for dataset in ["fashion","cifar10"]:
    staged=load_data(dataset,Path("/kaggle/working/data"),100)
    del staged
print("Datasets staged",flush=True)


In [ ]:
from concurrent.futures import ThreadPoolExecutor
workers=min(2,torch.cuda.device_count())
def shard(worker):
    for dataset in ["digits","fashion","cifar10"]:
        seeds=list(range(100+worker,110,workers))
        subprocess.run([sys.executable,str(base/"offline_benchmark.py"),"--dataset",dataset,"--seeds",*map(str,seeds),"--device",f"cuda:{worker}","--output",str(base/dataset/f"shard{worker}")],check=True)
with ThreadPoolExecutor(max_workers=workers) as pool:
    list(pool.map(shard,range(workers)))


In [ ]:
subprocess.run([sys.executable,str(base/"analyze_offline_benchmark.py"),str(base),"--output",str(base/"analysis")],check=True)


In [ ]:
(base/"COMPLETE.json").write_text(json.dumps(manifest,indent=2))
print("GPU benchmark complete",flush=True)
